In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Audio, display

In [ ]:
DATASET_NAME = "alstdi"  # options: "alstdi", "parkceleb"

real_metadata_path = Path(f"../data/metadata/{DATASET_NAME}_metadata.csv")
synthetic_metadata_path = Path(f"../outputs/generated_audio/{DATASET_NAME}_lflow/synthetic_metadata.csv")

real_metadata_path, synthetic_metadata_path

In [ ]:
real_df = pd.read_csv(real_metadata_path)
syn_df = pd.read_csv(synthetic_metadata_path)

real_df["source"] = "real"
syn_df["source"] = "synthetic"

print(f"Real recordings: {len(real_df)}")
print(f"Synthetic recordings: {len(syn_df)}")

real_df.head(), syn_df.head()

In [ ]:
subject_counts = syn_df.groupby("subject_id").size().sort_values(ascending=False)
subject_id = subject_counts.index[0]

print(f"Selected subject: {subject_id}")

In [ ]:
real_sub = real_df[real_df["subject_id"].astype(str) == str(subject_id)].sort_values("timestamp")
syn_sub = syn_df[syn_df["subject_id"].astype(str) == str(subject_id)].sort_values("timestamp")

plt.figure(figsize=(8, 4))

plt.plot(
    real_sub["timestamp"],
    real_sub["label"],
    marker="o",
    linestyle="-",
    label="Real recordings",
)

plt.scatter(
    syn_sub["timestamp"],
    syn_sub["label"],
    marker="o",
    facecolors="none",
    label="Synthetic samples",
)

plt.xlabel("Timestamp")
plt.ylabel("Normalized progression label")
plt.title(f"Real and Synthetic Trajectory: Subject {subject_id}")
plt.legend()
plt.show()

In [ ]:
example_subjects = (
    syn_df.groupby("subject_id")
    .size()
    .sort_values(ascending=False)
    .head(4)
    .index
)

for sid in example_subjects:
    real_sub = real_df[real_df["subject_id"].astype(str) == str(sid)].sort_values("timestamp")
    syn_sub = syn_df[syn_df["subject_id"].astype(str) == str(sid)].sort_values("timestamp")

    plt.figure(figsize=(8, 3))

    plt.plot(
        real_sub["timestamp"],
        real_sub["label"],
        marker="o",
        linestyle="-",
        label="Real",
    )

    plt.scatter(
        syn_sub["timestamp"],
        syn_sub["label"],
        marker="o",
        facecolors="none",
        label="Synthetic",
    )

    plt.xlabel("Timestamp")
    plt.ylabel("Progression label")
    plt.title(f"Subject {sid}")
    plt.legend()
    plt.show()

In [ ]:
sample_rows = syn_sub.head(3)

for _, row in sample_rows.iterrows():
    print("Subject:", row["subject_id"])
    print("Timestamp:", row["timestamp"])
    print("Interpolated label:", row["label"])
    print("Generated audio:", row["audio_path"])
    
    if pd.notna(row["audio_path"]):
        display(Audio(row["audio_path"]))

In [ ]:
sample = syn_df.dropna(subset=["audio_path", "reference_audio_path"]).iloc[0]

print("Subject:", sample["subject_id"])
print("Synthetic timestamp:", sample["timestamp"])
print("Synthetic label:", sample["label"])

print("\nReference audio:")
display(Audio(sample["reference_audio_path"]))

print("\nGenerated audio:")
display(Audio(sample["audio_path"]))

In [ ]:
predicted_scores_path = Path(f"../outputs/metrics/progression/{DATASET_NAME}_lflow_predicted_scores.csv")

if predicted_scores_path.exists():
    score_df = pd.read_csv(predicted_scores_path)

    plt.figure(figsize=(5, 5))
    plt.scatter(score_df["label"], score_df["predicted_score"], alpha=0.7)
    plt.xlabel("Interpolated progression label")
    plt.ylabel("Predicted severity score")
    plt.title("Progression Consistency")
    plt.show()
else:
    print("Predicted score file not found. Skipping progression consistency scatter.")